# Workshop: Tunix-Med · Part 2 — Baseline Evaluation

Before we fine-tune, we must establish a **quantitative starting point**. This notebook evaluates the *unmodified* Gemma 3 270M model on the same cardiology Q&A dataset that will be used for training in Part 3. Every result produced here becomes the **baseline** against which the fine-tuned model is compared in Part 4.

The core insight driving this evaluation strategy: a model’s out-of-the-box performance on specialist medical questions is a good proxy for how much “domain gap” exists between its general pre-training knowledge and the target domain. A large gap justifies supervised fine-tuning; a small gap suggests the model already generalises well and may only need a lightweight adaptation.

---

## Evaluation pipeline overview

We use a **three-metric composite** to assess answer quality, each measuring a different dimension:

| Metric | What it measures | Weight in final score |
|--------|------------------|-----------------------|
| **Keyword F1 (TF-IDF weighted)** | Presence of domain-specific medical terminology | 10% |
| **Semantic Similarity** | Meaning-level overlap via sentence embeddings | 30% |
| **AI Judge (Qwen 2.5 7B)** | Clinical accuracy and reasoning quality | 60% |

A fourth diagnostic, **Corpus Perplexity**, is computed separately. It measures how “surprised” the model is by the ground-truth answers and is used to track domain knowledge acquisition across parts 2 and 4.

The intentional weighting (60% AI Judge) reflects the core lesson of the workshop: automated text-matching metrics capture surface form, not clinical correctness. A model that paraphrases a wrong diagnosis with eloquent vocabulary would score highly on keyword and semantic metrics but poorly on the AI Judge.


In [1]:
import os
import warnings
import logging
import math

import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")
logging.getLogger("httpx").setLevel(logging.WARNING)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def info_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


device = info_device()
dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    else torch.float32
)
print(f"Device: {device}  |  dtype: {dtype}")

Device: cuda  |  dtype: torch.bfloat16


## 1 · Load Base Model & Dataset

We load the **Gemma 3 270M instruction-tuned** checkpoint directly from the Hugging Face Hub with no modifications — no adapters, no fine-tuning, no prompt engineering beyond the standard system message.

To ensure the evaluation is **reproducible and fair**, we use a fixed random seed (`SEED = 42`) to sample exactly 100 questions from the held-out 10% evaluation split. The same seed and sampling logic are reused in Part 4, guaranteeing that the baseline and final evaluations are compared on *identical* questions.

> **Why 270M and not a larger model?** The workshop deliberately chooses a small model to make the effect of fine-tuning more visible. A 7B model may already have absorbed some cardiology knowledge from its pre-training corpus; the 270M model’s domain gap is much more pronounced, making the before/after comparison more instructive.


### Deep Dive: Chat Templates and Prompt Format

Modern instruction-tuned LLMs are not plain language models — they are trained on **conversation datasets** formatted with special tokens that delimit roles (system, user, assistant). Failing to replicate this format at evaluation time is one of the most common sources of poor benchmark results.

#### What `apply_chat_template` does
The HuggingFace tokeniser’s `apply_chat_template` method inserts the model-specific control tokens in the correct positions. For Gemma 3, this produces something like:

```
<bos><start_of_turn>user
You are a knowledgeable medical assistant...
What is the first-line treatment for stable angina?<end_of_turn>
<start_of_turn>model
```

The final `<start_of_turn>model\n` is the **generation prompt** — the cue that tells the model it is now expected to produce the assistant’s response.

#### `add_generation_prompt=True`
When set to `True`, the template appends the assistant role opening tag so that `model.generate()` immediately starts generating the response, rather than predicting whether to continue the conversation or not.

#### `return_dict=True` and `return_tensors="pt"`
These flags cause `apply_chat_template` to return a HuggingFace `BatchEncoding` dict (with `input_ids` and `attention_mask` keys) ready to be unpacked with `**` into `model.generate()`. Without them, you would need to call `tokenizer(text, return_tensors="pt")` as a second step.


In [2]:
MODEL_NAME = "google/gemma-3-270m-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
    device_map="auto",
)
model.eval()

DATASET_ID = "lmassaron/medical-cardiology-qa"
full_ds = load_dataset(DATASET_ID, split="train")
n = len(full_ds)

# Sample 100 questions from the 10% eval split
EVAL_SPLIT, SEED, N_EVAL_QS = 0.1, 42, 100
rng = np.random.default_rng(SEED)
all_idx = rng.permutation(n)
eval_idx = all_idx[int(n * (1.0 - EVAL_SPLIT)) :]


def extract_qa(example):
    msgs = example["messages"]
    return {
        "question": next(m["content"] for m in msgs if m["role"] == "user"),
        "answer": next(m["content"] for m in msgs if m["role"] == "assistant"),
    }


rng2, qa_pairs, seen_prefixes = np.random.default_rng(SEED + 1), [], set()
for idx in rng2.permutation(eval_idx):
    if len(qa_pairs) >= N_EVAL_QS:
        break
    ex = extract_qa(full_ds[int(idx)])
    if len(ex["answer"]) < 25:
        continue
    prefix = " ".join(ex["question"].lower().split()[:4])
    if prefix in seen_prefixes:
        continue
    seen_prefixes.add(prefix)
    qa_pairs.append({"question": ex["question"], "answer": ex["answer"]})

data = pd.DataFrame(qa_pairs)
print(f"Sampled {len(data)} questions.")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu130).


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Sampled 100 questions.


## 2 · Baseline Inference & Corpus Perplexity

For each of the 100 evaluation questions, we run two forward passes:

1. **Generation pass** — `model.generate()` with greedy decoding (`do_sample=False`) to produce a deterministic answer. We apply a mild `repetition_penalty=1.1` to prevent the model from degenerating into repetitive phrases — a common failure mode in small models asked specialist questions they have not been trained on.

2. **Perplexity pass** — a separate forward pass with the *ground-truth* answer appended. The model sees the full prompt+answer sequence but loss is computed only on the answer tokens (prompt tokens are masked to `-100`, which PyTorch’s cross-entropy ignores). This gives us a per-example loss which is accumulated into a **corpus-level perplexity**.

The two passes are necessary because perplexity must be computed on the *reference* answer, not the generated one — otherwise a confidently wrong model would appear to have low perplexity.

> **Memory note**: the base model is deleted and GPU cache is cleared at the end of this cell before the judge model is loaded. On a single 24 GB GPU, holding both simultaneously would cause an OOM error.


### Deep Dive: Perplexity (PPL) as a Domain-Knowledge Metric

#### Mathematical definition
Given a sequence of tokens $w_1, w_2, \ldots, w_T$, perplexity is:

$$\text{PPL} = \exp\!\left(- \frac{1}{T} \sum_{t=1}^{T} \log P(w_t \mid w_1, \ldots, w_{t-1})\right)$$

This is simply $e^{\text{mean cross-entropy loss}}$. PyTorch’s `CrossEntropyLoss` (used internally by `model(input_ids, labels=labels)`) computes the mean over non-masked tokens, so we recover PPL as `math.exp(loss.item())`.

#### Why corpus-level PPL is more robust than per-example PPL
A simple average of per-example PPLs is biased: a single very short answer (few tokens, easy to predict) can inflate the average because its weight is identical to a long, complex answer. **Corpus PPL** corrects for this by weighting each answer proportionally to its token count:

$$\text{PPL}_{\text{corpus}} = \exp\!\left(\frac{\sum_i L_i \cdot N_i}{\sum_i N_i}\right)$$

where $L_i$ is the per-token loss for example $i$ and $N_i$ is its answer token count. This is the standard definition used in most LLM evaluation papers.

#### Interpreting the baseline PPL
- **PPL > 20**: the base model is largely unfamiliar with cardiology terminology and treatment protocols.
- **PPL 10–20**: some relevant knowledge exists but is incomplete or inconsistently applied.
- **PPL < 10**: the model already has strong domain coverage (rare for a 270M model on specialist text).

After SFT in Part 3, we expect to see a significant PPL reduction — the concrete numerical gap is the quantitative proof of domain knowledge acquisition.


In [3]:
SYSTEM_PROMPT = "You are a knowledgeable medical assistant specializing in cardiology. Answer clinical questions accurately, focusing on diagnostic criteria, treatment guidelines, and pathophysiology."
results_list = []

total_loss_bits = 0.0
total_tokens = 0

for _, row in tqdm(data.iterrows(), total=len(data), desc="Baseline Inference"):
    # 1. Generation
    encoded = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["question"]},
        ],
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)
    prompt_len = encoded["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(
            **encoded,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
    generated = tokenizer.decode(outputs[0, prompt_len:], skip_special_tokens=True).strip()

    # 2. Perplexity (True Corpus PPL)
    full_ids_ppl = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["answer"]},
        ],
        return_tensors="pt",
        return_dict=True,
    )["input_ids"].to(device)
    labels = full_ids_ppl.clone()
    labels[:, :prompt_len] = -100
    with torch.no_grad():
        out = model(full_ids_ppl, labels=labels)
        loss_i = out.loss.item()

    n_answer_tokens = (labels != -100).sum().item()
    total_loss_bits += loss_i * n_answer_tokens
    total_tokens += n_answer_tokens

    perplexity_i = math.exp(min(loss_i, 20.0))
    results_list.append(
        {
            "question": row["question"],
            "expected_answer": row["answer"],
            "generated_answer": generated,
            "perplexity": perplexity_i,
        }
    )

results_df = pd.DataFrame(results_list)
corpus_ppl = math.exp(total_loss_bits / total_tokens)
print(f"Inference complete. Corpus Perplexity: {corpus_ppl:.2f}")

del model
torch.cuda.empty_cache()

Baseline Inference:   0%|          | 0/100 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Inference complete. Corpus Perplexity: 21.68


## 3 · Scoring: Three-Metric Composite

The scoring pipeline applies three complementary metrics to each (generated, reference) pair and combines them into a single `final_score`. Using multiple metrics is important because each captures a different failure mode:

- A model that generates *factually correct but differently worded* answers will score high on semantic similarity but low on keyword F1.
- A model that *copies the question back* might score high on keyword overlap but low on semantic similarity and AI Judge.
- A model that produces *confident-sounding but clinically wrong* answers may score well on both text-based metrics but will be correctly penalised by the AI Judge.

The metrics are computed in three cells: keyword F1 and semantic similarity first (lightweight, CPU-friendly), then the AI Judge (requires loading a 7B model).


### Deep Dive: The Three Metrics

#### 1. TF-IDF Weighted Keyword F1
Standard F1 treats all words equally. A model that mentions “the” and “patient” overlapping with a reference answer gets the same credit as one that correctly uses “atrial fibrillation” and “rate control”. **TF-IDF weighting** fixes this: words that appear frequently across all 100 reference answers (common medical filler) get downweighted; rare, domain-specific terms get upweighted. The TF-IDF vectoriser is fitted on the reference answer corpus, so the IDF values reflect the actual vocabulary distribution of our cardiology dataset.

#### 2. Semantic Similarity via Sentence Embeddings
We use `all-MiniLM-L6-v2`, a 22M-parameter sentence transformer, to encode both the generated and reference answers into dense vector representations. Cosine similarity between these vectors captures *meaning-level* overlap, independent of wording. Crucially, raw cosine scores are **min-max normalised** within the batch before being used as a score, so all values are in [0, 1] relative to the spread of the current evaluation run.

#### 3. LLM-as-a-Judge (Qwen 2.5 7B Instruct)
Traditional metrics like ROUGE or BERTScore were designed for summarisation and translation tasks; they struggle with clinical Q&A where the *interpretation* of terminology matters as much as its presence. We use **Qwen 2.5 7B Instruct** loaded in **4-bit NF4 quantisation** (via BitsAndBytes) as an expert judge. The judge receives a structured prompt with:
- A 1–10 scoring rubric anchored at specific quality levels.
- The question, reference answer, and generated answer.
- An instruction to first write reasoning, then emit `Score: [number]` on the final line.

The score is parsed from the last few lines of the judge’s output with a regex fallback. This “chain of thought then score” approach is a well-established LLM-judging technique that produces more reliable scores than asking for a number directly.

#### 4-bit NF4 quantisation
Loading a 7B model in `float16` requires ~14 GB of VRAM. 4-bit NF4 quantisation (Normal Float 4, introduced in QLoRA) compresses each weight to 4 bits using a non-uniform quantisation grid optimised for normally distributed values, reducing memory to ~4 GB with minimal accuracy loss. `bnb_4bit_use_double_quant=True` adds a second quantisation step on the quantisation constants themselves, saving a further ~0.4 GB.


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
import re

_tfidf = TfidfVectorizer(analyzer="word", token_pattern=r"\b\w{4,}\b", sublinear_tf=True)
_tfidf.fit(results_df["expected_answer"].tolist())
_vocab, _idf = _tfidf.vocabulary_, _tfidf.idf_


def keyword_f1_tfidf(gen, exp):
    ref_kws = set(re.findall(r"\b\w{4,}\b", exp.lower()))
    gen_kws = set(re.findall(r"\b\w{4,}\b", gen.lower()))
    if not ref_kws:
        return 1.0

    def weighted_count(kws, universe):
        return sum(
            _idf[_vocab[w]] if w in _vocab else 1.0 for w in universe if w in kws
        )

    ref_weight = sum(_idf[_vocab[w]] if w in _vocab else 1.0 for w in ref_kws)
    gen_weight = (
        sum(_idf[_vocab[w]] if w in _vocab else 1.0 for w in gen_kws) if gen_kws else 0.0
    )
    if ref_weight == 0 or gen_weight == 0:
        return 0.0
    recall = weighted_count(gen_kws, ref_kws) / ref_weight
    precision = weighted_count(ref_kws, gen_kws) / gen_weight
    return float(
        (2 * precision * recall / (precision + recall))
        if (precision + recall) > 0
        else 0.0
    )


from sentence_transformers import SentenceTransformer, util

sim_model = SentenceTransformer("all-MiniLM-L6-v2")


def _raw_semantic(gen, exp):
    e1 = sim_model.encode(gen, convert_to_tensor=True, show_progress_bar=False)
    e2 = sim_model.encode(exp, convert_to_tensor=True, show_progress_bar=False)
    return float(util.pytorch_cos_sim(e1, e2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
judge_mdl = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL, quantization_config=bnb_config, device_map="auto"
)


def ai_judge(question, generated, expected):
    prompt = (
        "You are an expert clinical cardiologist and medical education evaluator.\n"
        "Evaluate the 'Generated Answer' against the 'Reference Answer' for the given 'Question'.\n\n"
        "### SCORING RUBRIC (1-10):\n"
        "- 1: NO ANSWER (e.g., greetings, refusals, 'I am ready to help') or COMPLETELY WRONG.\n"
        "- 2-4: MAJOR ERRORS (significant inaccuracies or misses the key point).\n"
        "- 5-6: PARTIALLY CORRECT (covers some aspects but lacks depth or minor inaccuracies).\n"
        "- 7-8: MOSTLY CORRECT (aligns with the reference, but slightly incomplete or verbose).\n"
        "- 9-10: EXCELLENT (clinically precise, factual, accurate, and matches or improves on the reference).\n\n"
        "### INSTRUCTIONS:\n"
        "1. Penalize non-answers or conversational fluff with a score of 1.\n"
        "2. Score ONLY the medical accuracy and clinical relevance.\n"
        "3. Ignore chatbot niceties (e.g., 'I hope this helps').\n"
        "4. First write reasoning, then on the last line write ONLY: 'Score: [number]'\n\n"
        f"Question: {question}\n"
        f"Reference Answer: {expected}\n"
        f"Generated Answer: {generated}\n"
    )
    inp = judge_tok.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(judge_mdl.device)
    with torch.no_grad():
        out = judge_mdl.generate(**inp, max_new_tokens=150, do_sample=False)
    txt = judge_tok.decode(out[0, inp["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    
    # Improved extraction: look for "Score: X" on the last few lines
    lines = txt.splitlines()
    for line in reversed(lines):
        m = re.search(r"Score:\s*(\d+)", line)
        if m:
            return max(min(int(m.group(1)) / 10.0, 1.0), 0.1)
            
    # Fallback
    m = re.search(r"\b(\d+)\b", txt.splitlines()[-1])
    return max(min((int(m.group(1)) / 10.0) if m else 0.5, 1.0), 0.1)

results_df["keyword_score"] = results_df.apply(
    lambda r: keyword_f1_tfidf(r["generated_answer"], r["expected_answer"]), axis=1
)
results_df["_raw_sim"] = results_df.apply(
    lambda r: _raw_semantic(r["generated_answer"], r["expected_answer"]), axis=1
)
sim_min, sim_max = results_df["_raw_sim"].min(), results_df["_raw_sim"].max()
results_df["semantic_score"] = (
    (results_df["_raw_sim"] - sim_min) / (sim_max - sim_min)
).clip(0, 1)

print("Running AI Judge...")
scores = []
for _, r in tqdm(results_df.iterrows(), total=len(results_df), desc="AI Judge"):
    scores.append(ai_judge(r["question"], r["generated_answer"], r["expected_answer"]))
results_df["ai_judge_score"] = scores
results_df["final_score"] = results_df["keyword_score"] * 0.1 + results_df["semantic_score"] * 0.3 + results_df["ai_judge_score"] * 0.6

print("\n--- BASELINE RESULTS ---")
print(f"  Mean Keyword Score  : {results_df['keyword_score'].mean():.3f}")
print(f"  Mean Semantic Score : {results_df['semantic_score'].mean():.3f}")
print(f"  Mean AI Judge Score : {results_df['ai_judge_score'].mean():.3f}")
print(f"  Mean Final Score    : {results_df['final_score'].mean():.3f}")
print(f"  Corpus Perplexity   : {corpus_ppl:.2f}")
results_df.to_csv("medical_baseline_results.csv", index=False)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Running AI Judge...


AI Judge:   0%|          | 0/100 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- BASELINE RESULTS ---
  Mean Keyword Score  : 0.208
  Mean Semantic Score : 0.620
  Mean AI Judge Score : 0.169
  Mean Final Score    : 0.308
  Corpus Perplexity   : 21.68


## 4 · Qualitative Analysis: Best & Worst Performers

Aggregate metrics tell us *how much* the model knows on average, but qualitative inspection tells us *what kind* of knowledge it has and where it fails. This section surfaces the three questions where the AI Judge gave the **highest** scores and the three where it gave the **lowest**.

### What to look for in the baseline
- **High-scoring baseline examples** tend to be questions where the answer requires general biomedical knowledge that was likely well-represented in the pre-training corpus (e.g., definitions of common conditions, basic pathophysiology).
- **Low-scoring baseline examples** tend to be questions requiring specific clinical protocols, drug dosing details, or rare condition management — the kind of granular procedural knowledge that only appears in specialist medical literature.

After viewing these examples, note the patterns. In Part 4 we will compare the same question types to see whether fine-tuning addressed the specific failure modes identified here.


In [6]:
def display_examples(df, title, n=3, ascending=False):
    print(f"\n{'='*80}\n{title}\n{'='*80}")
    subset = df.sort_values("ai_judge_score", ascending=ascending).head(n)
    for i, (_, row) in enumerate(subset.iterrows()):
        print(f"\n[{i+1}] Score: {row['ai_judge_score']:.1f} | Question: {row['question']}")
        print(f"\nGenerated Answer:\n{row['generated_answer']}")
        print(f"\nReference Answer:\n{row['expected_answer']}")
        print(f"\n{'-'*80}")

display_examples(results_df, "TOP 3 PERFORMING EXAMPLES (Baseline)", n=3, ascending=False)
display_examples(results_df, "BOTTOM 3 PERFORMING EXAMPLES (Baseline)", n=3, ascending=True)


TOP 3 PERFORMING EXAMPLES (Baseline)

[1] Score: 0.7 | Question: Can mitral valve prolapse cause symptoms beyond palpitations and chest pain?

Generated Answer:
Yes, mitral valve prolapse can cause symptoms beyond palpitations and chest pain.

Reference Answer:
Yes, mitral valve prolapse can cause various symptoms including fatigue, dizziness, and syncope. These symptoms can vary in severity and frequency among individuals.

--------------------------------------------------------------------------------

[2] Score: 0.7 | Question: Is there any notable difference in the etiology of constrictive pericarditis between Western and developing countries?

Generated Answer:
Yes, there is a notable difference in the etiology of constrictive pericarditis between Western and developing countries.

Reference Answer:
Yes, in developing countries, tuberculosis is frequently identified as the leading cause of constrictive pericarditis, which is less common in Western countries.

-------------------

## 5 · Save Results for Comparative Analysis

The results DataFrame is saved to `medical_baseline_results.csv`. This file serves as the **comparison anchor** loaded in notebook 04. The CSV contains one row per question with all metric scores, making it easy to compute delta statistics (e.g., mean improvement in AI Judge score) and to plot overlaid score distributions for visual comparison.

> **Important**: the `display_examples` function here uses the same logic as in notebook 04. Consistency in display format makes it easier to visually compare individual answers side-by-side when reviewing the workshop results.


In [7]:
def display_examples(df, title, n=3, ascending=False):
    header = "=" * 80
    divider = "-" * 80
    print(f"\n{header}\n{title}\n{header}")
    subset = df.sort_values("ai_judge_score", ascending=ascending).head(n)
    for i, (_, row) in enumerate(subset.iterrows()):
        print(f"\n[{i+1}] Score: {row['ai_judge_score']:.1f} | Question: {row['question']}")
        print(f"\nGenerated Answer:\n{row['generated_answer']}")
        print(f"\nReference Answer:\n{row['expected_answer']}")
        print(f"\n{divider}")

display_examples(results_df, "TOP 3 PERFORMING EXAMPLES (Baseline)", n=3, ascending=False)
display_examples(results_df, "BOTTOM 3 PERFORMING EXAMPLES (Baseline)", n=3, ascending=True)



TOP 3 PERFORMING EXAMPLES (Baseline)

[1] Score: 0.7 | Question: Can mitral valve prolapse cause symptoms beyond palpitations and chest pain?

Generated Answer:
Yes, mitral valve prolapse can cause symptoms beyond palpitations and chest pain.

Reference Answer:
Yes, mitral valve prolapse can cause various symptoms including fatigue, dizziness, and syncope. These symptoms can vary in severity and frequency among individuals.

--------------------------------------------------------------------------------

[2] Score: 0.7 | Question: Is there any notable difference in the etiology of constrictive pericarditis between Western and developing countries?

Generated Answer:
Yes, there is a notable difference in the etiology of constrictive pericarditis between Western and developing countries.

Reference Answer:
Yes, in developing countries, tuberculosis is frequently identified as the leading cause of constrictive pericarditis, which is less common in Western countries.

-------------------